# CS-781 Phase 4 - Court Judgment Prediction (CJPE) on our own Phase-1 models
**Settings:** Accelerator **GPU T4 x2**, Internet **On**, secret **HF_TOKEN** switched ON. Then **Save Version -> Save & Run All**.

What runs (everything is uploaded to `<you>/cs781-p1-ckpt/phase4/`; finished runs are skipped if you re-run):
1. Cell 3 - downloads the official `Phase_4_CJPE` data once, normalises + tokenises it with our Phase-1 tokenizer, caches it on HF (~15 min first time).
2. Cell 7 - a 3-step safety test on the real data, then 15 experiments on both GPUs (each has a time budget; total about 4.5-5 h):
   all four Phase-1 models; LoRMA (compulsory), full fine-tuning, BitFit, bottleneck adapters, partial (top-layer) fine-tuning,
   frozen probe, knowledge distillation (llama teacher -> shared student); long-document study: last 512 tokens, first 512,
   first+last, hierarchical last 4 chunks, hierarchical 4 chunks spread over the whole judgment; class weighting on/off.
3. Cell 8 - results table, confusion matrices, false-positive / false-negative analysis, a cross-check of every run with the
   OFFICIAL `evaluate_phase4.py` on dev, all zips in `/kaggle/working/zips`.
4. Cell 9 - ensembles (`ens_top3`, `ens_top5`, `ens_greedy`).

Dev labels are used only to pick the checkpoint and the decision threshold - never for training.


In [1]:
# ===================== P4 CELL 1 - Setup + download the Phase-4 (CJPE) data and our tokenizer =====================
# Settings: Accelerator = GPU T4 x2, Internet = On, Secret "HF_TOKEN" (or "HF") switched ON for this notebook.
import os, sys, json, time, math, random, re, shutil, collections, subprocess
os.environ["HF_HOME"] = "/tmp/hf_home"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download
_sec = UserSecretsClient()
try:
    HF_TOKEN = _sec.get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = _sec.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER   = api.whoami()["name"]
DATA_REPO = f"{HF_USER}/cs781-p1-data"          # our tokenizer
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"          # Phase-1 models; Phase-4 outputs go to its phase4/ folder
SRC_REPO  = "Exploration-Lab/CS-781-Capstone"
WORK      = "/kaggle/working"
P4        = "/tmp/cs781/p4"; os.makedirs(P4, exist_ok=True)
RAW       = f"{P4}/flat"; os.makedirs(RAW, exist_ok=True)
DATA      = f"{P4}/data"; os.makedirs(DATA, exist_ok=True)       # token cache (train/dev/test.npz)

# ---- our tokenizer (Phase 1) ----
TOK_PATH = hf_hub_download(DATA_REPO, "tokenizer.json", repo_type="dataset", token=HF_TOKEN, local_dir=f"{P4}/tok")
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOK_PATH)
print("tokenizer vocab:", tok.get_vocab_size())

import torch
print(f"torch {torch.__version__} | GPUs {torch.cuda.device_count()}",
      *[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


tokenizer.json:   0%|          | 0.00/2.32M [00:00<?, ?B/s]

tokenizer vocab: 32768
torch 2.11.0+cu128 | GPUs 2 Tesla T4 Tesla T4


In [2]:
%%writefile /kaggle/working/model.py
"""CS-781 Phase 1 - four legal language-model architectures, written from scratch in PyTorch.

  gpt2    : classic decoder-only Transformer (learned absolute positions, LayerNorm, GELU MLP, multi-head attention)
  llama   : modern decoder-only Transformer (RoPE, RMSNorm, SwiGLU, grouped-query attention, no biases)
  shared  : lightweight decoder (ALBERT-style factorised embeddings + cycle cross-layer weight sharing + multi-query attn)
  encdec  : encoder-decoder Transformer (bidirectional encoder over the prompt, causal decoder with cross-attention)

All models: tied input/output embeddings, pre-norm residual blocks, PyTorch SDPA attention, KV-cache generation.
"""
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    arch: str = "llama"
    vocab_size: int = 32768
    d_model: int = 768
    n_layer: int = 12            # decoder-only: layers; encdec: layers in EACH of encoder and decoder; shared: effective depth
    n_head: int = 12
    n_kv_head: int = 4           # GQA groups (llama/encdec); shared uses 1 (MQA); gpt2 uses n_head (MHA)
    d_ff: int = 2304
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    dropout: float = 0.0
    n_unique_layers: int = 6     # shared only: number of distinct blocks, reused cyclically
    d_embed: int = 256           # shared only: factorised embedding size
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


def preset(arch: str, **overrides) -> ModelConfig:
    base = {
        "gpt2":   dict(arch="gpt2", d_model=768, n_layer=12, n_head=12, n_kv_head=12, d_ff=3072),
        "llama":  dict(arch="llama", d_model=768, n_layer=12, n_head=12, n_kv_head=4, d_ff=2304),
        "shared": dict(arch="shared", d_model=768, n_layer=12, n_head=12, n_kv_head=1, d_ff=3072,
                       n_unique_layers=6, d_embed=256),
        "encdec": dict(arch="encdec", d_model=768, n_layer=6, n_head=12, n_kv_head=4, d_ff=2304),
    }[arch]
    base.update(overrides)
    return ModelConfig(**base)


# ----------------------------------------------------------------------------------------------------------------
# building blocks
# ----------------------------------------------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.ln = nn.LayerNorm(d, eps=eps)

    def forward(self, x):
        return F.layer_norm(x.float(), self.ln.normalized_shape, self.ln.weight.float(), self.ln.bias.float(),
                            self.ln.eps).to(x.dtype)


def rope_cache(head_dim, max_len, theta, device=None):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, hd/2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (T, hd/2) - rotate pairs (x[..., :hd/2], x[..., hd/2:])
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    """Multi-head / grouped-query / multi-query attention. Self-attention (optionally causal) or cross-attention."""

    def __init__(self, cfg: ModelConfig, n_kv_head: int, bias: bool, use_rope: bool, cross: bool = False):
        super().__init__()
        d, h = cfg.d_model, cfg.n_head
        assert d % h == 0 and h % n_kv_head == 0
        self.h, self.kv, self.hd = h, n_kv_head, d // h
        self.use_rope, self.cross = use_rope, cross
        self.q = nn.Linear(d, h * self.hd, bias=bias)
        self.k = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.v = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.o = nn.Linear(h * self.hd, d, bias=bias)
        self.o.is_residual_proj = True
        self.dropout = cfg.dropout

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.hd).transpose(1, 2)          # (B, n, T, hd)

    def _expand(self, x):
        return x if self.kv == self.h else x.repeat_interleave(self.h // self.kv, dim=1)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None):
        """
        x: (B, T, d).  rope: (cos, sin) full tables.  pos: absolute position of x[:,0] (for RoPE during decoding).
        cache: dict used during generation. Self-attn: keys 'k','v' accumulate. Cross-attn: 'k','v' computed once.
        memory: encoder output (B, S, d) for cross-attention.
        """
        q = self._split(self.q(x), self.h)
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k(memory), self.kv), self._split(self.v(memory), self.kv)
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
        else:
            k, v = self._split(self.k(x), self.kv), self._split(self.v(x), self.kv)
            if self.use_rope:
                cos, sin = rope
                T = x.shape[1]
                q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
                k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2); v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            # causal mask only needed when queries and keys cover the same span (training / prefill)
            is_causal = causal and q.shape[2] == k.shape[2] and q.shape[2] > 1
            if causal and q.shape[2] != k.shape[2] and q.shape[2] != 1:
                raise ValueError("multi-token decode with a non-empty cache is not supported")
        y = F.scaled_dot_product_attention(q, self._expand(k), self._expand(v), is_causal=is_causal,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], self.h * self.hd)
        return self.o(y)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig, kind: str, bias: bool):
        super().__init__()
        self.kind = kind
        if kind == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        else:
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=bias)
        self.down.is_residual_proj = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x), approximate="tanh")
        return self.drop(self.down(h))


class Block(nn.Module):
    """Pre-norm Transformer block: x + Attn(Norm(x)) [+ CrossAttn(Norm(x), memory)] + MLP(Norm(x))."""

    def __init__(self, cfg: ModelConfig, style: str, n_kv_head: int, cross: bool = False):
        super().__init__()
        gpt2 = style == "gpt2"
        Norm = LayerNorm if gpt2 else RMSNorm
        self.n1 = Norm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg, n_kv_head, bias=gpt2, use_rope=not gpt2)
        self.has_cross = cross
        if cross:
            self.nc = Norm(cfg.d_model, cfg.norm_eps)
            self.xattn = Attention(cfg, n_kv_head, bias=False, use_rope=False, cross=True)
        self.n2 = Norm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg, "gelu" if gpt2 else "swiglu", bias=gpt2)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None, xcache=None):
        x = x + self.drop(self.attn(self.n1(x), rope=rope, pos=pos, causal=causal, cache=cache))
        if self.has_cross:
            x = x + self.drop(self.xattn(self.nc(x), memory=memory, cache=xcache))
        return x + self.mlp(self.n2(x))


# ----------------------------------------------------------------------------------------------------------------
# the language model
# ----------------------------------------------------------------------------------------------------------------
class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        a = cfg.arch
        assert a in ("gpt2", "llama", "shared", "encdec"), a
        d = cfg.d_model
        if a == "shared":
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_embed)
            self.emb_proj = nn.Linear(cfg.d_embed, d, bias=False)          # E -> d   (factorised embedding)
        else:
            self.tok_emb = nn.Embedding(cfg.vocab_size, d)
        if a == "gpt2":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, d)
        self.drop = nn.Dropout(cfg.dropout)

        if a == "gpt2":
            self.blocks = nn.ModuleList([Block(cfg, "gpt2", cfg.n_head) for _ in range(cfg.n_layer)])
        elif a == "llama":
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
        elif a == "shared":
            assert cfg.n_layer % cfg.n_unique_layers == 0
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_unique_layers)])
        else:  # encdec
            self.enc_blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
            self.enc_norm = RMSNorm(d, cfg.norm_eps)
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head, cross=True) for _ in range(cfg.n_layer)])
        self.norm = LayerNorm(d, cfg.norm_eps) if a == "gpt2" else RMSNorm(d, cfg.norm_eps)

        if a != "gpt2":
            hd = d // cfg.n_head
            cos, sin = rope_cache(hd, max(2 * cfg.max_seq_len, 2048), cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        n_res = 2 * self.n_layer_applications() + (cfg.n_layer if a == "encdec" else 0)
        for name, p in self.named_parameters():
            if name.endswith("o.weight") or name.endswith("down.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    # -- helpers ------------------------------------------------------------------------------------------------
    def n_layer_applications(self):
        return self.cfg.n_layer * (2 if self.cfg.arch == "encdec" else 1)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _rope(self):
        return None if self.cfg.arch == "gpt2" else (self.rope_cos, self.rope_sin)

    def _embed(self, idx, pos=0):
        x = self.tok_emb(idx)
        if self.cfg.arch == "shared":
            x = self.emb_proj(x)
        if self.cfg.arch == "gpt2":
            T = idx.shape[1]
            if pos + T > self.cfg.max_seq_len:
                raise ValueError(f"gpt2 positions exceed max_seq_len ({pos + T} > {self.cfg.max_seq_len})")
            x = x + self.pos_emb(torch.arange(pos, pos + T, device=idx.device))[None]
        return self.drop(x)

    def _logits(self, h):
        h = self.norm(h)
        if self.cfg.arch == "shared":
            h = F.linear(h, self.emb_proj.weight.t())     # d -> E   (tied with the input projection)
        return F.linear(h, self.tok_emb.weight)            # tied output embedding

    def _decoder_layers(self):
        if self.cfg.arch == "shared":                       # cycle sharing: 0,1,..,5,0,1,..,5
            return [self.blocks[i % self.cfg.n_unique_layers] for i in range(self.cfg.n_layer)]
        return list(self.blocks)

    def encode(self, src):
        x = self._embed(src)
        for blk in self.enc_blocks:
            x = blk(x, rope=self._rope(), pos=0, causal=False)
        return self.enc_norm(x)

    # -- training forward -----------------------------------------------------------------------------------------
    def forward(self, idx, targets=None, src=None, loss_mask=None):
        """Decoder-only: idx (B,T) -> logits; loss vs targets (B,T).  encdec: src (B,S) is the encoder input."""
        memory = self.encode(src) if self.cfg.arch == "encdec" else None
        x = self._embed(idx)
        for blk in self._decoder_layers():
            x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
        logits = self._logits(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        if loss_mask is not None:
            m = loss_mask.reshape(-1).float()
            return logits, (loss * m).sum() / m.sum().clamp(min=1)
        return logits, loss.mean()

    # -- generation -----------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0,
                 no_repeat_ngram=0, banned_ids=(), bos_id=2, generator=None, use_cache=True):
        """prompt: (B, T) LongTensor, all rows the same length (no padding). Returns (B, max_new_tokens) new ids.
        temperature 0 = greedy."""
        B = prompt.shape[0]
        dev = prompt.device
        layers = self._decoder_layers()
        if self.cfg.arch == "encdec":
            memory = self.encode(prompt)
            seq = torch.full((B, 1), bos_id, dtype=torch.long, device=dev)   # decoder starts from <|bos|>
            history = prompt                                                   # for repetition penalties
        else:
            memory, seq, history = None, prompt, prompt
        caches = [dict() for _ in layers] if use_cache else None
        xcaches = [dict() for _ in layers] if (use_cache and memory is not None) else [None] * len(layers)
        out = []
        cur, pos = seq, 0
        for step in range(max_new_tokens):
            if use_cache:
                x = self._embed(cur, pos=pos)
                for li, blk in enumerate(layers):
                    x = blk(x, rope=self._rope(), pos=pos, causal=True, cache=caches[li], memory=memory,
                            xcache=xcaches[li])
                pos += cur.shape[1]
            else:
                x = self._embed(seq)
                for blk in layers:
                    x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
            logits = self._logits(x[:, -1:, :])[:, -1, :].float()
            full_hist = torch.cat([history] + out, dim=1) if out else history
            nxt = sample_next(logits, full_hist, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram,
                              banned_ids, generator)
            out.append(nxt)
            seq = torch.cat([seq, nxt], dim=1)
            cur = nxt
        return torch.cat(out, dim=1)


def sample_next(logits, history, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram, banned_ids,
                generator=None):
    """logits (B,V) float32, history (B,L) long (prompt + generated so far). Returns (B,1)."""
    logits = logits.clone()
    if banned_ids:
        logits[:, list(banned_ids)] = -float("inf")
    if repetition_penalty != 1.0:
        # CTRL-style penalty on tokens already present in the recent history (last 256 tokens)
        recent = history[:, -256:]
        sc = torch.gather(logits, 1, recent)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, recent, sc)
    if no_repeat_ngram and history.shape[1] >= no_repeat_ngram:
        n = no_repeat_ngram
        hist = history[:, -512:].tolist()
        for b, h in enumerate(hist):
            prefix = tuple(h[-(n - 1):]) if n > 1 else tuple()
            banned = {tuple(h[i:i + n])[-1] for i in range(len(h) - n + 1) if tuple(h[i:i + n - 1]) == prefix}
            if banned:
                logits[b, list(banned)] = -float("inf")
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = torch.softmax(sl, dim=-1).cumsum(dim=-1)
        remove = cp - torch.softmax(sl, dim=-1) > top_p          # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)


Writing /kaggle/working/model.py


In [3]:
%%writefile /kaggle/working/p4data.py
"""CS-781 Phase 4 (CJPE): read train/dev/test once, normalise like our Phase-1 text, tokenise with OUR tokenizer,
and cache token ids (uint16) per split -> <out_dir>/<split>.npz.  Run once; training scripts only read the cache."""
import argparse, hashlib, json, os, re, sys, time, unicodedata
import numpy as np

# same punctuation map as Phase-1 pre-processing (copied from p2text.PUNCT)
PUNCT = {"‘": "'", "’": "'", "‚": "'", "‛": "'", "′": "'",
         "“": '"', "”": '"', "„": '"', "″": '"',
         "‐": "-", "‑": "-", "‒": "-", "–": "-", "—": "-",
         "―": "-", "−": "-", "•": "*", "·": "."}
_TABLE = str.maketrans(PUNCT)
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_WS = re.compile(r"\s+")


def normalize(t):
    """Fast version of p2text.normalize_with_map (no character map needed here): fancy punctuation -> ASCII,
    NFKC, accents stripped, other non-ASCII dropped, control characters dropped, whitespace runs -> one space."""
    if not isinstance(t, str):
        t = " ".join(t) if isinstance(t, list) else str(t or "")
    t = t.translate(_TABLE)
    if not t.isascii():
        t = unicodedata.normalize("NFKC", t)
        t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = _CTRL.sub("", t)
    return _WS.sub(" ", t).strip()


def load_records(path):
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, dict):                       # tolerate {id: {...}} or {"data": [...]}
        if "data" in d and isinstance(d["data"], list):
            d = d["data"]
        else:
            d = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in d.items()]
    out = []
    for r in d:
        lab = r.get("label", r.get("labels", r.get("decision")))
        if isinstance(lab, list):
            lab = lab[0] if lab else None
        out.append((str(r.get("id", r.get("ID", r.get("name")))), r.get("text", r.get("document", "")),
                    None if lab is None else int(lab)))
    return out


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--raw_dir", required=True, help="folder with train.json, dev.json, test.json")
    p.add_argument("--tok", required=True)
    p.add_argument("--out_dir", required=True)
    a = p.parse_args(argv)
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(a.tok)
    os.makedirs(a.out_dir, exist_ok=True)
    stats, hashes = {}, {}
    for split in ["train", "dev", "test"]:
        t0 = time.time()
        recs = load_records(os.path.join(a.raw_dir, f"{split}.json"))
        n_raw = len(recs)
        seen, keep, dup_same, dup_conflict = {}, [], 0, 0
        for did, text, lab in recs:                    # one entry per case id (the brief: one prediction per test id)
            if did in seen:
                if seen[did] == lab:
                    dup_same += 1
                else:
                    dup_conflict += 1
                continue
            seen[did] = lab
            keep.append((did, text, lab))
        del recs
        ids_all, offs, labels, names, nchars, h = [], [0], [], [], [], []
        B = 500
        for s in range(0, len(keep), B):
            part = keep[s:s + B]
            texts = [normalize(t) for _, t, _ in part]
            encs = tok.encode_batch(texts, add_special_tokens=False)
            for (did, _, lab), txt, e in zip(part, texts, encs):
                arr = np.asarray(e.ids, dtype=np.uint16)
                ids_all.append(arr); offs.append(offs[-1] + len(arr))
                labels.append(-1 if lab is None else lab); names.append(did); nchars.append(len(txt))
                h.append(hashlib.md5(txt.encode()).hexdigest())
        ids_cat = np.concatenate(ids_all) if ids_all else np.zeros(0, np.uint16)
        lens = np.diff(np.asarray(offs))
        np.savez(os.path.join(a.out_dir, f"{split}.npz"), ids=ids_cat, off=np.asarray(offs, np.int64),
                 label=np.asarray(labels, np.int64), name=np.asarray(names), nchars=np.asarray(nchars, np.int64))
        hashes[split] = h
        if split == "dev":          # local truth file in the format of the official evaluate_phase4.py
            json.dump([{"id": n, "gold": int(l)} for n, l in zip(names, labels)], open(os.path.join(a.out_dir, "dev_truth.json"), "w"))
        lab_arr = np.asarray(labels)
        st = {"records_in_file": n_raw, "unique_ids": len(keep), "duplicate_ids_same_label": dup_same,
              "duplicate_ids_conflicting_label": dup_conflict,
              "label_counts": {str(k): int((lab_arr == k).sum()) for k in (0, 1)} if split != "test" else None,
              "tokens_mean": float(lens.mean()) if len(lens) else 0, "tokens_median": float(np.median(lens)) if len(lens) else 0,
              "tokens_p90": float(np.percentile(lens, 90)) if len(lens) else 0, "tokens_max": int(lens.max()) if len(lens) else 0,
              "frac_over_512": float((lens > 512).mean()) if len(lens) else 0,
              "frac_over_2048": float((lens > 2048).mean()) if len(lens) else 0,
              "empty_docs": int((lens == 0).sum()), "seconds": round(time.time() - t0, 1)}
        stats[split] = st
        print(split, json.dumps(st), flush=True)
    tr = set(hashes["train"])
    stats["text_overlap"] = {"dev_in_train": sum(x in tr for x in hashes["dev"]),
                             "test_in_train": sum(x in tr for x in hashes["test"]),
                             "test_in_dev": len(set(hashes["test"]) & set(hashes["dev"]))}
    print("identical texts:", stats["text_overlap"], flush=True)
    json.dump(stats, open(os.path.join(a.out_dir, "stats.json"), "w"), indent=1)


if __name__ == "__main__":
    main()


Writing /kaggle/working/p4data.py


In [4]:
%%writefile /kaggle/working/p4cj.py
"""CS-781 Phase 4: Court Judgment Prediction (binary 0/1) on our own Phase-1 models.

Model  = Phase-1 backbone (+ adaptation method) -> one vector per chunk (mean of token states)
         -> document vector (attention pooling over the chunks of a judgment) -> 2-way classifier.
Long documents ("rep"):
  last        : the last W tokens of the judgment (one chunk)
  first       : the first W tokens (one chunk)
  firstlast   : first W/2 + last W/2 tokens (one chunk)
  hier_tail   : the last K chunks of W tokens (hierarchical: K chunk vectors -> learned attention pooling)
  hier_spread : K chunks of W tokens spread evenly over the WHOLE judgment (hierarchical)
Methods: lorma (LoRMA, compulsory) | full | bitfit | adapter (bottleneck) | partial (top layers) | head (frozen probe);
         --kd_teacher <run> adds knowledge distillation from that run's saved train probabilities.
Dev is used only for choosing the checkpoint and the decision threshold (calibration), never for training.
"""
import argparse, json, math, os, random, sys, time, zipfile
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, ModelConfig                      # our Phase-1 model code (unchanged)


# ================================================================================================================
# adaptation modules (our own code)
# ================================================================================================================
class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel & Modi, 2025), additive rank inflation: W' = (I + (alpha/r) B A) W0.
    y0 = W0 x ; y = y0 + s * B(A y0) (+ bias).  B = 0 -> starts exactly at W0."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        d_out = base.out_features
        self.A = nn.Parameter(torch.randn(r, d_out) / math.sqrt(d_out))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class BiasLinear(nn.Module):
    """BitFit on a bias-free layer: frozen W, a new trainable bias that starts at 0 (so training starts at W0)."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.bias = nn.Parameter(torch.zeros(base.out_features))

    def forward(self, x):
        return F.linear(x, self.base.weight, self.bias)


class AdapterOut(nn.Module):
    """Bottleneck adapter (Houlsby et al. 2019 / Pfeiffer placement) on a sub-layer's output projection:
    y = W x ; y + Up(GELU(Down(y))).  Up = 0 at start -> starts exactly at the Phase-1 model."""

    def __init__(self, base: nn.Linear, dim=64):
        super().__init__()
        self.base = base
        d = base.out_features
        self.a_down = nn.Linear(d, dim)
        self.a_up = nn.Linear(dim, d)
        nn.init.zeros_(self.a_up.weight); nn.init.zeros_(self.a_up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.a_up(F.gelu(self.a_down(y)))


# ================================================================================================================
# the CJPE network
# ================================================================================================================
class CJNet(nn.Module):
    def __init__(self, lm: LegalLM, bidir=True, dropout=0.1):
        super().__init__()
        self.lm, self.arch, self.bidir = lm, lm.cfg.arch, bidir
        d = lm.cfg.d_model
        if self.arch == "encdec":                       # only the (bidirectional) encoder is used
            lm.blocks = nn.ModuleList()
        self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))   # learned pooling over chunks
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(d, 2)

    def layers(self):
        if self.arch == "encdec":
            return list(self.lm.enc_blocks)
        return self.lm._decoder_layers()

    def features(self, idx):
        lm = self.lm
        if self.arch == "encdec":
            return lm.encode(idx)
        x = lm._embed(idx)
        for blk in lm._decoder_layers():
            x = blk(x, rope=lm._rope(), pos=0, causal=not self.bidir)
        return lm.norm(x)

    def pool(self, chunk_vecs, owner, n_docs):
        """chunk_vecs (N,d) float32, owner (N,) doc index -> (n_docs, d) by attention over each doc's chunks."""
        s = self.att(chunk_vecs).squeeze(-1)                                     # (N,)
        s = s - s.max()
        w = torch.exp(s)
        den = torch.zeros(n_docs, device=s.device).index_add_(0, owner, w)
        doc = torch.zeros(n_docs, chunk_vecs.shape[1], device=s.device).index_add_(0, owner, w[:, None] * chunk_vecs)
        return doc / den.clamp(min=1e-12)[:, None]

    def head(self, doc_vecs):
        return self.out(self.drop(doc_vecs)).float()


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


def setup_method(net, a):
    """Freeze / wrap according to the method. Returns (backbone_params_at_lr, adapter_params_at_peft_lr)."""
    lm = net.lm
    for p in net.parameters():
        p.requires_grad = True
    m = a["method"]
    if m == "full":
        return
    for p in lm.parameters():
        p.requires_grad = False
    root = lm.enc_blocks if net.arch == "encdec" else lm.blocks
    if m == "lorma":
        tg = set(a["targets"].split(","))
        n = 0
        for mod in list(root.modules()):
            for name, ch in list(mod.named_children()):
                if isinstance(ch, nn.Linear) and name in tg:
                    setattr(mod, name, LoRMALinear(ch, a["rank"], a["alpha"], 0.05)); n += 1
        print(f"lorma: wrapped {n} linear layers", flush=True)
    elif m == "bitfit":
        n = 0
        for mod in list(root.modules()):
            for name, ch in list(mod.named_children()):
                if isinstance(ch, nn.Linear):
                    if ch.bias is None:
                        setattr(mod, name, BiasLinear(ch))
                    else:
                        ch.bias.requires_grad = True
                    n += 1
        for mod in lm.modules():                                   # norm gains / biases are "bias-like" terms too
            if type(mod).__name__ in ("RMSNorm", "LayerNorm", "LayerNorm_"):
                for p in mod.parameters():
                    p.requires_grad = True
        print(f"bitfit: biases on {n} linear layers + all norm parameters", flush=True)
    elif m == "adapter":
        n = 0
        for mod in list(root.modules()):
            for name, ch in list(mod.named_children()):
                if isinstance(ch, nn.Linear) and name in ("o", "down"):
                    setattr(mod, name, AdapterOut(ch, a["adapter_dim"])); n += 1
        print(f"adapter: {n} bottleneck adapters (dim {a['adapter_dim']})", flush=True)
    elif m == "partial":
        blocks = list(root)
        top = blocks[-a["partial_layers"]:]
        for b in top:
            for p in b.parameters():
                p.requires_grad = True
        norm = lm.enc_norm if net.arch == "encdec" else lm.norm
        for p in norm.parameters():
            p.requires_grad = True
        print(f"partial: top {len(top)} of {len(blocks)} blocks + final norm trainable", flush=True)
    elif m == "head":
        print("head: backbone frozen, only pooling + classifier train", flush=True)
    else:
        raise ValueError(m)


# ================================================================================================================
# data
# ================================================================================================================
class Split:
    def __init__(self, path, limit=0):
        z = np.load(path, allow_pickle=False)
        self.ids, self.off = z["ids"], z["off"]
        self.label, self.name = z["label"], [str(x) for x in z["name"]]
        n = len(self.name) if not limit else min(limit, len(self.name))
        self.n = n

    def doc(self, i):
        return self.ids[self.off[i]:self.off[i + 1]]


def chunks_of(ids, rep, W, K):
    """-> list of np arrays (token ids) for one judgment. Every chunk has length W unless the whole doc is shorter."""
    n = len(ids)
    if n == 0:
        return [np.zeros(1, np.uint16)]
    if rep == "last":
        return [ids[-W:]]
    if rep == "first":
        return [ids[:W]]
    if rep == "firstlast":
        return [ids] if n <= W else [np.concatenate([ids[:W // 2], ids[n - (W - W // 2):]])]
    if n <= W:
        return [ids]
    if rep == "hier_tail":
        if n >= K * W:
            return [ids[n - (K - j) * W: n - (K - j - 1) * W] for j in range(K)]
        k = math.ceil(n / W)
    elif rep == "hier_spread":
        k = min(K, math.ceil(n / W))
    else:
        raise ValueError(rep)
    starts = np.linspace(0, n - W, k).round().astype(int)
    return [ids[s:s + W] for s in starts]


def doc_logits(net, chunk_lists, device, amp, max_chunks=32, return_vec=False):
    """chunk_lists: one list of chunks per document -> logits (B, 2). Chunks of equal length are batched together
    (our Phase-1 models have no padding mask, so sequences of different lengths are never mixed)."""
    flat = [(di, c) for di, cl in enumerate(chunk_lists) for c in cl]
    by_len = defaultdict(list)
    for j, (_, c) in enumerate(flat):
        by_len[len(c)].append(j)
    vecs = [None] * len(flat)
    for L, js in by_len.items():
        for s in range(0, len(js), max_chunks):
            part = js[s:s + max_chunks]
            x = torch.from_numpy(np.stack([flat[j][1] for j in part]).astype(np.int64)).to(device)
            with amp():
                h = net.features(x)
            v = h.float().mean(1)
            for k, j in enumerate(part):
                vecs[j] = v[k]
    V = torch.stack(vecs)
    owner = torch.tensor([di for di, _ in flat], device=device)
    doc = net.pool(V, owner, len(chunk_lists))
    return (net.head(doc), doc) if return_vec else net.head(doc)


@torch.no_grad()
def predict(net, split, idxs, a, device, amp):
    net.eval()
    out = np.zeros(len(idxs), np.float32)
    docs_per = max(1, a.eval_chunks // (1 if a.rep in ("last", "first", "firstlast") else a.chunks))
    for s in range(0, len(idxs), docs_per):
        part = idxs[s:s + docs_per]
        lg = doc_logits(net, [chunks_of(split.doc(i), a.rep, a.win, a.chunks) for i in part], device, amp,
                        max_chunks=a.eval_chunks)
        out[s:s + len(part)] = torch.softmax(lg, -1)[:, 1].cpu().numpy()
    net.train()
    return out


# ================================================================================================================
# metric (official primary metric: macro-F1 over the two classes; precision / recall secondary)
# ================================================================================================================
def scores(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    out = {}
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        out[c] = (p, r, f, int((y == c).sum()))
    # same definitions as the official evaluate_phase4.py: macro-F1 over {0,1}; precision / recall of class 1 (binary)
    return {"f1": (out[0][2] + out[1][2]) / 2, "p": out[1][0], "r": out[1][1],
            "p_macro": (out[0][0] + out[1][0]) / 2, "r_macro": (out[0][1] + out[1][1]) / 2,
            "acc": float((y == pred).mean()) if len(y) else 0.0, "per_class": {str(c): out[c] for c in (0, 1)},
            "confusion": [[int(((y == t) & (pred == q)).sum()) for q in (0, 1)] for t in (0, 1)]}


THRESHOLDS = [round(x, 2) for x in np.arange(0.05, 0.951, 0.01)]


def best_threshold(y, p1):
    """Calibration on dev: the decision threshold on P(label=1) that maximises dev macro-F1 (ties -> closest to 0.5)."""
    return max(THRESHOLDS, key=lambda t: (scores(y, (p1 >= t).astype(int))["f1"], -abs(t - 0.5)))


# ================================================================================================================
def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--run_name", required=True)
    p.add_argument("--base_run", required=True)
    p.add_argument("--base_file", default="best_model.pt")
    p.add_argument("--method", default="lorma", choices=["lorma", "full", "bitfit", "adapter", "partial", "head"])
    p.add_argument("--kd_teacher", default="", help="Phase-4 run whose train probabilities are the teacher")
    p.add_argument("--kd_alpha", type=float, default=0.5, help="weight of the gold-label loss (1-alpha: teacher)")
    p.add_argument("--kd_T", type=float, default=2.0)
    p.add_argument("--save_train_probs", type=int, default=0, help="1 = also predict all train docs (KD teacher)")
    p.add_argument("--bidir", type=int, default=1)
    p.add_argument("--rep", default="last", choices=["last", "first", "firstlast", "hier_tail", "hier_spread"])
    p.add_argument("--win", type=int, default=512)
    p.add_argument("--chunks", type=int, default=4, help="K for the hierarchical representations")
    p.add_argument("--bs", type=int, default=16, help="chunks per training step")
    p.add_argument("--eval_chunks", type=int, default=48)
    p.add_argument("--epochs", type=float, default=2.0)
    p.add_argument("--max_train_min", type=float, default=40.0, help="time budget for training (min)")
    p.add_argument("--evals", type=int, default=6, help="dev evaluations during training")
    p.add_argument("--lr", type=float, default=3e-5, help="backbone LR (full / partial)")
    p.add_argument("--peft_lr", type=float, default=5e-4, help="LoRMA / adapter / BitFit LR")
    p.add_argument("--head_lr", type=float, default=1e-3)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--alpha", type=float, default=32)
    p.add_argument("--targets", default="q,k,v,o")
    p.add_argument("--adapter_dim", type=int, default=64)
    p.add_argument("--partial_layers", type=int, default=4)
    p.add_argument("--wd", type=float, default=0.01)
    p.add_argument("--warmup", type=float, default=0.06)
    p.add_argument("--class_weight", default="balanced", choices=["none", "balanced"])
    p.add_argument("--seed", type=int, default=13)
    p.add_argument("--data_dir", required=True, help="folder with train.npz / dev.npz / test.npz from p4data.py")
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--max_steps", type=int, default=-1, help="debug")
    p.add_argument("--train_limit", type=int, default=0, help="debug: use only the first N train docs")
    p.add_argument("--dev_limit", type=int, default=0, help="debug")
    return p.parse_args(argv)


def hf_up(repo, local, remote):
    if not repo:
        return
    from huggingface_hub import HfApi
    for k in range(3):
        try:
            HfApi(token=os.environ.get("HF_TOKEN")).upload_file(path_or_fileobj=local, path_in_repo=remote,
                                                                 repo_id=repo, repo_type="model")
            return
        except Exception as e:
            print("[hf] upload failed:", e, flush=True); time.sleep(15 * (k + 1))


def hf_get(repo, remote, local_dir):
    if not repo:
        return None
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo, remote, repo_type="model", local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def main(argv=None):
    args = parse_args(argv)
    t_start = time.time()
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cuda = dev.type == "cuda"
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if cuda else (lambda: torch.autocast("cpu", enabled=False))
    run_dir = os.path.join(args.out_dir, args.run_name); os.makedirs(run_dir, exist_ok=True)
    logf = open(os.path.join(run_dir, "log.txt"), "a")

    def log(*m):
        s = f"[{time.strftime('%H:%M:%S')}][{args.run_name}] " + " ".join(str(x) for x in m)
        print(s, flush=True); logf.write(s + "\n"); logf.flush()

    # ---------------- data ----------------
    tr = Split(os.path.join(args.data_dir, "train.npz"), args.train_limit)
    dv = Split(os.path.join(args.data_dir, "dev.npz"), args.dev_limit)
    te = Split(os.path.join(args.data_dir, "test.npz"))
    tr_idx = [i for i in range(tr.n) if tr.label[i] in (0, 1)]
    dv_idx = list(range(dv.n)); te_idx = list(range(te.n))
    y_tr = np.array([tr.label[i] for i in tr_idx]); y_dv = np.array([dv.label[i] for i in dv_idx])
    log(f"train {len(tr_idx)} docs (label 1: {y_tr.mean():.3f}) | dev {len(dv_idx)} (label 1: {y_dv.mean():.3f}) | test {len(te_idx)}"
        f" | rep {args.rep} W {args.win} K {args.chunks}")

    # ---------------- model ----------------
    ck_path = os.path.join(args.ckpt_dir, "runs", args.base_run, args.base_file)
    if not os.path.exists(ck_path):
        hf_get(args.hf_repo, f"runs/{args.base_run}/{args.base_file}", args.ckpt_dir)
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    lm_cfg = ck["config"]
    if lm_cfg["arch"] == "gpt2":                                  # learned positions: never exceed the trained length
        args.win = min(args.win, lm_cfg["max_seq_len"])
    lm = LegalLM(ModelConfig(**lm_cfg))
    missing, _ = lm.load_state_dict(ck["model"], strict=False)
    bad = [k for k in missing if not k.startswith("blocks.")] if lm_cfg["arch"] == "encdec" else list(missing)
    assert not bad, f"missing backbone weights: {bad[:5]}"
    del ck
    net = CJNet(lm, bidir=bool(args.bidir))
    setup_method(net, vars(args))
    net.to(dev)
    total, trainable = count_params(net)
    log(f"backbone {args.base_run} ({lm_cfg['arch']}) | method {args.method}{' + KD from ' + args.kd_teacher if args.kd_teacher else ''}"
        f" | bidir {args.bidir} | params {total / 1e6:.2f}M trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")

    teacher = None
    if args.kd_teacher:
        tp = os.path.join(args.out_dir, args.kd_teacher, "train_probs.npz")
        if not os.path.exists(tp):
            q = hf_get(args.hf_repo, f"phase4/runs/{args.kd_teacher}/train_probs.npz", os.path.join(args.out_dir, "_hub"))
            assert q is not None, f"teacher {args.kd_teacher}: train_probs.npz not found"
            tp = q
        z = np.load(tp)
        tmap = dict(zip([str(x) for x in z["name"]], z["p1"]))
        teacher = np.array([tmap[tr.name[i]] for i in tr_idx], np.float32)
        log(f"KD teacher {args.kd_teacher}: {len(teacher)} train probabilities, alpha {args.kd_alpha} T {args.kd_T}")

    # ---------------- optimiser ----------------
    head_params = list(net.att.parameters()) + list(net.out.parameters())
    hp = set(id(p) for p in head_params)
    other = [(n, p) for n, p in net.named_parameters() if p.requires_grad and id(p) not in hp]
    if args.method in ("full", "partial"):
        groups = [{"params": [p for _, p in other], "lr": args.lr}]
    else:
        groups = [{"params": [p for _, p in other], "lr": args.peft_lr}]
    groups.append({"params": head_params, "lr": args.head_lr})
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=args.wd)
    base_lrs = [g["lr"] for g in groups]
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    if args.class_weight == "balanced":
        cnt = np.bincount(y_tr, minlength=2).astype(float)
        w = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=dev)
    else:
        w = None
    log("class weights:", None if w is None else [round(x, 3) for x in w.tolist()])

    # ---------------- batches: documents grouped so that each step sees about `bs` chunks ----------------
    single = args.rep in ("last", "first", "firstlast")
    per_doc = 1 if single else args.chunks
    docs_per_step = max(1, args.bs // per_doc)
    steps_per_epoch = math.ceil(len(tr_idx) / docs_per_step)
    total_steps = max(1, int(steps_per_epoch * args.epochs))
    if args.max_steps > 0:
        total_steps = min(total_steps, args.max_steps)
    warm = max(1, int(args.warmup * total_steps))
    budget = args.max_train_min * 60
    eval_every = max(1, total_steps // max(1, args.evals))
    log(f"{docs_per_step} docs/step | {steps_per_epoch} steps/epoch | planned {total_steps} steps | budget {args.max_train_min:.0f} min")

    rng = random.Random(args.seed)
    order = []
    best = {"f1": -1.0}
    best_state, step, t_train, eval_time = None, 0, time.time(), 0.0
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    trainable_names = [n for n, p in net.named_parameters() if p.requires_grad]

    def evaluate():
        nonlocal best_state, eval_time
        te0 = time.time()
        p1 = predict(net, dv, dv_idx, args, dev, amp)
        th = best_threshold(y_dv, p1)
        r_t = scores(y_dv, (p1 >= th).astype(int)); r_5 = scores(y_dv, (p1 >= 0.5).astype(int))
        eval_time += time.time() - te0
        log(f"step {step}/{total_steps} | dev macro-F1 {r_t['f1']:.4f} at threshold {th:.2f} (at 0.50: {r_5['f1']:.4f}) "
            f"acc {r_t['acc']:.4f} | {(time.time() - t_train) / 60:.1f} min")
        if r_t["f1"] > best["f1"]:
            best.update(f1=r_t["f1"], f1_05=r_5["f1"], th=th, step=step, p1=p1)
            sd = net.state_dict()
            best_state = {k: sd[k].detach().to("cpu", copy=True) for k in trainable_names}

    pos = {i: k for k, i in enumerate(tr_idx)}
    stop = False
    net.train()
    while not stop:
        if not order:
            order = tr_idx[:]; rng.shuffle(order)
        batch = order[:docs_per_step]; order = order[docs_per_step:]
        lr_mult = min(1.0, (step + 1) / warm) * (0.5 * (1 + math.cos(math.pi * min(1.0, step / total_steps))))
        for g, b in zip(opt.param_groups, base_lrs):
            g["lr"] = b * max(lr_mult, 0.02)
        y = torch.tensor([tr.label[i] for i in batch], dtype=torch.long, device=dev)
        logits = doc_logits(net, [chunks_of(tr.doc(i), args.rep, args.win, args.chunks) for i in batch], dev, amp,
                            max_chunks=max(args.bs, 1))
        loss = F.cross_entropy(logits, y, weight=w)
        if teacher is not None:
            T = args.kd_T
            pt = torch.tensor([teacher[pos[i]] for i in batch], dtype=torch.float32, device=dev).clamp(1e-6, 1 - 1e-6)
            q = torch.stack([1 - pt, pt], -1) ** (1.0 / T)
            q = q / q.sum(-1, keepdim=True)                                    # = softmax(teacher_logits / T)
            kd = F.kl_div(F.log_softmax(logits / T, -1), q, reduction="batchmean") * T * T
            loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
        scaler.step(opt); scaler.update()
        step += 1
        if step == 30 and args.max_steps <= 0:                    # fit the plan into the time budget
            sec = (time.time() - t_train) / step
            fit = int(max(60, (budget - 0.12 * budget) / sec))
            if fit < total_steps:
                log(f"{sec:.2f} s/step -> time budget allows {fit} of {total_steps} steps; schedule shortened")
                total_steps = fit; eval_every = max(1, total_steps // max(1, args.evals))
            else:
                log(f"{sec:.2f} s/step -> planned {total_steps} steps fit the budget")
        if step % 50 == 0:
            log(f"step {step}/{total_steps} loss {loss.item():.4f} | {(time.time() - t_train) / 60:.1f} min")
        if step % eval_every == 0 or step >= total_steps:
            evaluate()
        if step >= total_steps or (time.time() - t_train) > budget:
            if step % eval_every != 0 and step < total_steps:
                evaluate()
            stop = True
    train_min = (time.time() - t_train) / 60
    log(f"training done: {step} steps in {train_min:.1f} min (dev evaluation {eval_time / 60:.1f} min) | best step {best['step']}")

    # ---------------- best checkpoint -> dev / test (/ train) predictions ----------------
    sd = net.state_dict()
    sd.update(best_state)
    net.load_state_dict(sd)
    p_dev = best["p1"]; th = best["th"]
    t0 = time.time()
    p_test = predict(net, te, te_idx, args, dev, amp)
    test_s = time.time() - t0
    r = scores(y_dv, (p_dev >= th).astype(int)); r5 = scores(y_dv, (p_dev >= 0.5).astype(int))
    pred_test = (p_test >= th).astype(int)
    preds = [{"id": te.name[i], "pred": int(pred_test[k])} for k, i in enumerate(te_idx)]
    assert len(preds) == te.n and len(set(x["id"] for x in preds)) == te.n and all(x["pred"] in (0, 1) for x in preds)
    json.dump(preds, open(os.path.join(run_dir, "prediction.json"), "w"))
    with zipfile.ZipFile(os.path.join(run_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(os.path.join(run_dir, "prediction.json"), arcname="prediction.json")
    json.dump([{"id": dv.name[i], "pred": int(p_dev[k] >= th), "prob1": float(p_dev[k]), "label": int(y_dv[k])}
               for k, i in enumerate(dv_idx)], open(os.path.join(run_dir, "dev_prediction.json"), "w"))
    np.savez(os.path.join(run_dir, "probs.npz"), dev_name=np.array([dv.name[i] for i in dv_idx]), dev_p1=p_dev,
             test_name=np.array([te.name[i] for i in te_idx]), test_p1=p_test)
    if args.save_train_probs:
        t1 = time.time()
        p_tr = predict(net, tr, tr_idx, args, dev, amp)
        np.savez(os.path.join(run_dir, "train_probs.npz"), name=np.array([tr.name[i] for i in tr_idx]), p1=p_tr)
        log(f"train probabilities for KD saved ({(time.time() - t1) / 60:.1f} min; train macro-F1 at threshold "
            f"{th:.2f}: {scores(y_tr, (p_tr >= th).astype(int))['f1']:.4f})")
    torch.save({"state_dict": {k: v.half() if v.is_floating_point() else v for k, v in net.state_dict().items()},
                "args": vars(args), "lm_config": lm_cfg}, os.path.join(run_dir, "best.pt"))
    summary = {"run": args.run_name, "base_run": args.base_run, "arch": lm_cfg["arch"], "method": args.method,
               "kd_teacher": args.kd_teacher, "rep": args.rep, "win": args.win, "chunks": args.chunks,
               "class_weight": args.class_weight, "params_total": total, "params_trainable": trainable,
               "dev_f1": r["f1"], "dev_f1_at_0.5": r5["f1"], "threshold": th, "dev_p": r["p"], "dev_r": r["r"],
               "dev_acc": r["acc"], "dev_per_class": r["per_class"], "dev_confusion": r["confusion"],
               "best_step": best["step"], "steps": step, "train_minutes": round(train_min, 1),
               "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else 0.0,
               "infer_test_seconds": round(test_s, 1), "n_test": te.n, "test_pos_rate": float(pred_test.mean()),
               "n_train": len(tr_idx), "n_dev": len(dv_idx), "args": vars(args)}
    json.dump(summary, open(os.path.join(run_dir, "summary.json"), "w"), indent=1)
    log(f"DEV macro-F1 {r['f1']:.4f} (P {r['p']:.4f} R {r['r']:.4f}, acc {r['acc']:.4f}) at threshold {th:.2f} | "
        f"at 0.50: {r5['f1']:.4f} | confusion [[TN FP][FN TP]] {r['confusion']}")
    log(f"TEST: {te.n} cases, predicted 1 for {pred_test.mean():.3f} -> {run_dir}/submission.zip")
    if args.hf_repo:
        files = ["prediction.json", "submission.zip", "dev_prediction.json", "probs.npz", "log.txt", "best.pt"]
        if args.save_train_probs:
            files.append("train_probs.npz")
        for fn in files:
            hf_up(args.hf_repo, os.path.join(run_dir, fn), f"phase4/runs/{args.run_name}/{fn}")
        hf_up(args.hf_repo, os.path.join(run_dir, "summary.json"), f"phase4/runs/{args.run_name}/summary.json")   # last = done marker
    log(f"DONE in {(time.time() - t_start) / 60:.1f} min")


if __name__ == "__main__":
    main()


Writing /kaggle/working/p4cj.py


In [5]:
%%writefile /kaggle/working/p4ens.py
"""CS-781 Phase 4: ensembles of our fine-tuned CJPE models (all built on our own Phase-1 models).
Each run saved P(label=1) for every dev and test case (probs.npz). An ensemble = plain average of the members'
probabilities; the decision threshold is chosen on dev (calibration), exactly like a single model.
  ens_top3 / ens_top5 : the K runs with the best dev macro-F1 (no search -> least fitted to dev)
  ens_greedy          : greedy forward selection on dev
"""
import argparse, json, os, shutil, zipfile
import numpy as np

import p4cj as M


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--ks", default="3,5")
    p.add_argument("--min_dev", type=float, default=0.55)
    p.add_argument("--max_members", type=int, default=5)
    p.add_argument("--min_gain", type=float, default=0.002)
    p.add_argument("--tag", default="")
    a = p.parse_args(argv)
    dv = M.Split(os.path.join(a.data_dir, "dev.npz")); te = M.Split(os.path.join(a.data_dir, "test.npz"))
    y = np.array(dv.label[:dv.n]); dev_names, test_names = dv.name[:dv.n], te.name[:te.n]

    names = set(x for x in os.listdir(a.out_dir) if os.path.isdir(os.path.join(a.out_dir, x)))
    if a.hf_repo:
        from huggingface_hub import HfApi
        for f in HfApi(token=os.environ.get("HF_TOKEN")).list_repo_files(a.hf_repo):
            if f.startswith("phase4/runs/") and f.endswith("/summary.json"):
                names.add(f.split("/")[2])
    P, summ = {}, {}
    for n in sorted(names):
        if n.startswith("ens_") or n.startswith("_"):
            continue
        d = os.path.join(a.out_dir, n); os.makedirs(d, exist_ok=True)
        for fn in ["summary.json", "probs.npz"]:
            if not os.path.exists(os.path.join(d, fn)):
                q = M.hf_get(a.hf_repo, f"phase4/runs/{n}/{fn}", os.path.join(a.out_dir, "_hub"))
                if q:
                    shutil.copy(q, os.path.join(d, fn))
        if not (os.path.exists(os.path.join(d, "summary.json")) and os.path.exists(os.path.join(d, "probs.npz"))):
            continue
        s = json.load(open(os.path.join(d, "summary.json")))
        if "method" not in s or s["dev_f1"] < a.min_dev:
            continue
        z = np.load(os.path.join(d, "probs.npz"))
        dm = dict(zip([str(x) for x in z["dev_name"]], z["dev_p1"])); tm = dict(zip([str(x) for x in z["test_name"]], z["test_p1"]))
        if not all(k in dm for k in dev_names) or not all(k in tm for k in test_names):
            print("skip (ids do not match):", n); continue
        P[n] = (np.array([dm[k] for k in dev_names]), np.array([tm[k] for k in test_names]))
        summ[n] = s
    cands = sorted(P, key=lambda n: -summ[n]["dev_f1"])
    print(f"[ensemble] {len(cands)} candidates:", ", ".join(f"{n}={summ[n]['dev_f1']:.4f}" for n in cands), flush=True)
    if len(cands) < 2:
        print("[ensemble] fewer than 2 candidates"); return

    def dev_f1(members):
        pd = np.mean([P[m][0] for m in members], 0)
        th = M.best_threshold(y, pd)
        return M.scores(y, (pd >= th).astype(int)), th

    def write(name, members):
        r, th = dev_f1(members)
        pt = np.mean([P[m][1] for m in members], 0)
        pred = (pt >= th).astype(int)
        rd = os.path.join(a.out_dir, name); os.makedirs(rd, exist_ok=True)
        preds = [{"id": k, "pred": int(v)} for k, v in zip(test_names, pred)]
        assert len(preds) == len(set(test_names)) and all(x["pred"] in (0, 1) for x in preds)
        json.dump(preds, open(os.path.join(rd, "prediction.json"), "w"))
        with zipfile.ZipFile(os.path.join(rd, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(os.path.join(rd, "prediction.json"), arcname="prediction.json")
        info = {"members": members, "threshold": th, "dev_f1": r["f1"], "dev_p": r["p"], "dev_r": r["r"],
                "dev_acc": r["acc"], "dev_confusion": r["confusion"], "test_pos_rate": float(pred.mean())}
        json.dump(info, open(os.path.join(rd, "summary.json"), "w"), indent=1)
        print(f"[ensemble] {name}: {len(members)} models {members} | dev macro-F1 {r['f1']:.4f} at threshold {th:.2f} "
              f"| confusion {r['confusion']} -> {rd}/submission.zip", flush=True)
        if a.hf_repo:
            for fn in ["prediction.json", "submission.zip", "summary.json"]:
                M.hf_up(a.hf_repo, os.path.join(rd, fn), f"phase4/runs/{name}/{fn}")

    for K in [int(k) for k in a.ks.split(",") if k.strip()]:
        if K <= len(cands):
            write(f"ens_{a.tag}top{K}", cands[:K])
    chosen = [cands[0]]; cur = dev_f1(chosen)[0]["f1"]
    while len(chosen) < a.max_members:
        best_n, best_f = None, cur
        for n in cands:
            if n not in chosen:
                f = dev_f1(chosen + [n])[0]["f1"]
                if f > best_f:
                    best_n, best_f = n, f
        if best_n is None or best_f - cur < a.min_gain:
            break
        chosen.append(best_n); cur = best_f
    if len(chosen) > 1:
        write(f"ens_{a.tag}greedy", chosen)


if __name__ == "__main__":
    main()


Writing /kaggle/working/p4ens.py


In [6]:
%%writefile /kaggle/working/evaluate_phase4.py
#!/usr/bin/env python3
"""Local evaluator for Phase 4.

Truth JSON: a list of objects, [{"id":"...", "gold":0|1}, ...]
Prediction JSON: [{"id":"...", "pred":0|1}, ...]
"""
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out

def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 4 binary predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)


Writing /kaggle/working/evaluate_phase4.py


In [7]:
# ===================== P4 CELL 3 - Data: normalise + tokenise ONCE with our tokenizer (cached on Hugging Face) =====================
# First session: downloads the official Phase_4_CJPE files (~850 MB), tokenises them (~10-15 min) and uploads the cache.
# Later sessions: just downloads the cache (phase4/data/*.npz).
sys.path.insert(0, WORK)
hub_files = set(api.list_repo_files(CKPT_REPO))
cache = ["train.npz", "dev.npz", "test.npz", "stats.json", "dev_truth.json"]
if all(f"phase4/data/{f}" in hub_files for f in cache):
    for f in cache:
        shutil.copy(hf_hub_download(CKPT_REPO, f"phase4/data/{f}", repo_type="model", token=HF_TOKEN, local_dir=f"{P4}/hub"), f"{DATA}/{f}")
    print("token cache downloaded from Hugging Face")
else:
    t0 = time.time()
    for p in [x.path for x in api.list_repo_tree(SRC_REPO, repo_type="dataset", path_in_repo="Phase_4_CJPE", recursive=True)
              if getattr(x, "size", None) is not None]:
        lp = hf_hub_download(SRC_REPO, p, repo_type="dataset", token=HF_TOKEN, local_dir=f"{P4}/raw")
        shutil.copy(lp, f"{RAW}/{p.split('/')[-1]}")
    missing = [n for n in ["train.json", "dev.json", "test.json"] if not os.path.exists(f"{RAW}/{n}")]
    assert not missing, f"missing files {missing} - paste this output to Claude"
    print("Phase-4 files:", {f: f"{os.path.getsize(f'{RAW}/{f}') / 1e6:.0f} MB" for f in sorted(os.listdir(RAW))},
          f"| downloaded in {time.time() - t0:.0f}s")
    r = subprocess.run([sys.executable, f"{WORK}/p4data.py", "--raw_dir", RAW, "--tok", TOK_PATH, "--out_dir", DATA],
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    assert r.returncode == 0, "data preparation failed - paste this output to Claude"
    for f in cache:
        api.upload_file(path_or_fileobj=f"{DATA}/{f}", path_in_repo=f"phase4/data/{f}", repo_id=CKPT_REPO, repo_type="model")
    print(f"tokenised + uploaded in {(time.time() - t0) / 60:.1f} min")
    shutil.rmtree(f"{P4}/raw", ignore_errors=True); shutil.rmtree(RAW, ignore_errors=True)
stats = json.load(open(f"{DATA}/stats.json"))
for sp in ["train", "dev", "test"]:
    s = stats[sp]
    print(f"{sp:5s}: {s['unique_ids']:6d} cases (records {s['records_in_file']}, duplicate ids {s['duplicate_ids_same_label']}"
          f"+{s['duplicate_ids_conflicting_label']} conflicting) | labels {s['label_counts']} | tokens mean {s['tokens_mean']:.0f} "
          f"median {s['tokens_median']:.0f} p90 {s['tokens_p90']:.0f} max {s['tokens_max']} | >512: {100 * s['frac_over_512']:.0f}% "
          f">2048: {100 * s['frac_over_2048']:.0f}% | empty {s['empty_docs']}")
print("identical texts across splits:", stats["text_overlap"])


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

tokenised + uploaded in 4.1 min
train:  32305 cases (records 37387, duplicate ids 5082+0 conflicting) | labels {'0': 18920, '1': 13385} | tokens mean 3801 median 2449 p90 7736 max 489109 | >512: 88% >2048: 57% | empty 0
dev  :    994 cases (records 1988, duplicate ids 994+0 conflicting) | labels {'0': 497, '1': 497} | tokens mean 4471 median 3391 p90 8194 max 52771 | >512: 99% >2048: 78% | empty 0
test :   1517 cases (records 1517, duplicate ids 0+0 conflicting) | labels None | tokens mean 4590 median 3503 p90 8539 max 50737 | >512: 99% >2048: 78% | empty 0
identical texts across splits: {'dev_in_train': 0, 'test_in_train': 0, 'test_in_dev': 0}


In [8]:
# ===================== P4 CELL 7 - Run the experiments on both GPUs (queue with KD dependency, resumable) =====================
# Every run has a TIME BUDGET (max_train_min): it trains until the budget or the planned epochs, evaluating on dev
# several times and keeping the best checkpoint. Finished runs (phase4/runs/<run>/summary.json on HF) are skipped.
import threading
CK = "/tmp/cs781/ckpt"
OUT = f"{WORK}/p4runs"; os.makedirs(OUT, exist_ok=True)
for run, fn in [("dapt_llama", "best_model.pt"), ("explore_gpt2", "final_model.pt"),
                ("explore_encdec", "final_model.pt"), ("explore_shared", "final_model.pt")]:
    hf_hub_download(CKPT_REPO, f"runs/{run}/{fn}", repo_type="model", local_dir=CK, token=HF_TOKEN)
print("Phase-1 checkpoints ready")

COMMON = dict(win=512, chunks=4, bs=16, epochs=2, evals=6, lr=3e-5, peft_lr=5e-4, head_lr=1e-3, rank=16, alpha=32,
              class_weight="balanced", max_train_min=35)
LLAMA  = dict(base_run="dapt_llama", base_file="best_model.pt")
GPT2   = dict(base_run="explore_gpt2", base_file="final_model.pt", bidir=0)        # GPT-2: left-to-right (Phase 2 lesson)
ENCDEC = dict(base_run="explore_encdec", base_file="final_model.pt")
SHARED = dict(base_run="explore_shared", base_file="final_model.pt")
HIER = dict(epochs=1, evals=3, max_train_min=50)                                    # 4 chunks per case = 4x the work
RUNS = [   # in priority order: the required experiments first
    ("cj_llama_lorma_last",   {**LLAMA,  "method": "lorma", "rep": "last", "save_train_probs": 1}),   # LoRMA (compulsory) + KD teacher
    ("cj_llama_lorma_hier",   {**LLAMA,  "method": "lorma", "rep": "hier_tail", **HIER}),             # hierarchical: last 4 chunks
    ("cj_shared_kd_last",     {**SHARED, "method": "full", "lr": 5e-5, "rep": "last", "kd_teacher": "cj_llama_lorma_last", "max_train_min": 30}),  # KD
    ("cj_gpt2_lorma_last",    {**GPT2,   "method": "lorma", "rep": "last", "max_train_min": 30}),
    ("cj_encdec_lorma_last",  {**ENCDEC, "method": "lorma", "rep": "last", "max_train_min": 25}),
    ("cj_llama_full_last",    {**LLAMA,  "method": "full", "rep": "last"}),                         # full fine-tuning
    ("cj_llama_bitfit_last",  {**LLAMA,  "method": "bitfit", "peft_lr": 1e-3, "rep": "last", "max_train_min": 25}),
    ("cj_llama_adapter_last", {**LLAMA,  "method": "adapter", "peft_lr": 1e-3, "rep": "last", "max_train_min": 30}),
    ("cj_llama_partial_last", {**LLAMA,  "method": "partial", "partial_layers": 4, "lr": 5e-5, "rep": "last", "max_train_min": 30}),
    ("cj_llama_lorma_first",  {**LLAMA,  "method": "lorma", "rep": "first", "max_train_min": 25}),    # section study: beginning
    ("cj_llama_lorma_spread", {**LLAMA,  "method": "lorma", "rep": "hier_spread", **HIER}),           # whole judgment, 4 chunks
    ("cj_shared_full_last",   {**SHARED, "method": "full", "lr": 5e-5, "rep": "last", "max_train_min": 30}),   # same student WITHOUT KD
    ("cj_llama_lorma_firstlast", {**LLAMA, "method": "lorma", "rep": "firstlast", "max_train_min": 25}),
    ("cj_llama_head_last",    {**LLAMA,  "method": "head", "head_lr": 1e-3, "rep": "last", "max_train_min": 15}),   # frozen probe
    ("cj_llama_lorma_last_nocw", {**LLAMA, "method": "lorma", "rep": "last", "class_weight": "none", "max_train_min": 30}),
]
ONLY = None          # e.g. ["cj_llama_lorma_last"] to run a subset

def p4_cmd(name, cfg, out_dir, hf_repo, extra=()):
    cmd = [sys.executable, f"{WORK}/p4cj.py", "--run_name", name, "--data_dir", DATA, "--ckpt_dir", CK,
           "--out_dir", out_dir, "--hf_repo", hf_repo]
    for k, v in {**COMMON, **cfg}.items():
        cmd += [f"--{k}", str(v)]
    return cmd + list(extra)

# ---- safety test: 3 training steps + dev + FULL test prediction + zip on the REAL data; nothing uploaded ----
t_s = time.time()
smoke = p4_cmd("_smoke", {**SHARED, "method": "lorma", "rep": "hier_tail", "max_steps": 3, "evals": 1},
               f"{P4}/smoke", "", ["--train_limit", "64", "--dev_limit", "64"])
r = subprocess.run(smoke, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), capture_output=True, text=True)
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "][_smoke]" in l][-8:]))
if r.returncode != 0 or not os.path.exists(f"{P4}/smoke/_smoke/submission.zip"):
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
    raise RuntimeError("SAFETY TEST FAILED - nothing was trained. Send this output to Claude.")
print(f"safety test passed in {time.time() - t_s:.0f}s\n")
if torch.cuda.device_count() == 0:
    print("CPU session: switch Accelerator to GPU T4 x2 and run again."); RUNS = []

hub_files = set(api.list_repo_files(CKPT_REPO))
done, failed, running, lock = set(), set(), set(), threading.Lock()
pending = [(n, c) for n, c in RUNS if ONLY is None or n in ONLY]
for n, c in list(pending):
    if f"phase4/runs/{n}/summary.json" in hub_files:          # finished in an earlier session -> skip
        os.makedirs(f"{OUT}/{n}", exist_ok=True)
        for fn in ["summary.json", "submission.zip", "prediction.json", "dev_prediction.json", "probs.npz"]:
            if f"phase4/runs/{n}/{fn}" in hub_files:
                p = hf_hub_download(CKPT_REPO, f"phase4/runs/{n}/{fn}", repo_type="model", local_dir=f"{P4}/hub", token=HF_TOKEN)
                shutil.copy(p, f"{OUT}/{n}/{fn}")
        done.add(n); pending.remove((n, c)); print("already done:", n)

def next_job():
    with lock:
        for i, (n, c) in enumerate(pending):
            t = c.get("kd_teacher")
            if t and t not in done and f"phase4/runs/{t}/train_probs.npz" in hub_files:
                done.add(t)
            if t and t not in done and t not in running and all(t != m for m, _ in pending):
                failed.add(t)
            if t and t in failed:
                pending.pop(i); failed.add(n); print(f"SKIP {n}: teacher {t} failed / not available", flush=True); return "AGAIN"
            if not t or t in done:
                running.add(n); return pending.pop(i)
        return "WAIT" if pending else None

def worker(gpu):
    while True:
        job = next_job()
        if job is None: return
        if job in ("WAIT", "AGAIN"):
            time.sleep(30 if job == "WAIT" else 0); continue
        name, cfg = job
        with open(f"{OUT}/{name}.out", "w") as f:
            r = subprocess.run(p4_cmd(name, cfg, OUT, CKPT_REPO), env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                               stdout=f, stderr=subprocess.STDOUT)
        ok_ = r.returncode == 0 and os.path.exists(f"{OUT}/{name}/summary.json")
        with lock:
            (done if ok_ else failed).add(name); running.discard(name)
            print(f"{'FINISHED' if ok_ else 'FAILED  '} {name} on GPU {gpu}", flush=True)
            if not ok_:
                print(f"===== last lines of {name}.out =====\n" + "\n".join(open(f"{OUT}/{name}.out").read().splitlines()[-30:]), flush=True)

def status_line(name):
    p = f"{OUT}/{name}.out"
    if not os.path.exists(p): return "(waiting)"
    keep = [l for l in open(p).read().splitlines() if "step" in l or "DEV" in l or "Error" in l or "DONE" in l]
    return keep[-1][:170] if keep else "(starting)"

threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads: t.start()
t0 = time.time()
while any(t.is_alive() for t in threads):
    for t in threads: t.join(timeout=300 / len(threads))
    print(f"--- {(time.time() - t0) / 60:.0f} min | done {sorted(done)} | failed {sorted(failed)}", flush=True)
    for n, _ in RUNS:
        if n not in done and n not in failed and (ONLY is None or n in ONLY): print(f"   {n:26s} {status_line(n)}", flush=True)
print("ALL FINISHED. done:", sorted(done), "| failed:", sorted(failed))


runs/dapt_llama/best_model.pt: reconstructing file:   0%|          |  0.00B /  431MB            

runs/dapt_llama/best_model.pt: downloading bytes:           |  0.00B            

runs/explore_gpt2/final_model.pt: reconstructing file:   0%|          |  0.00B /  444MB            

runs/explore_gpt2/final_model.pt: downloading bytes:           |  0.00B            

runs/explore_encdec/final_model.pt: reconstructing file:   0%|          |  0.00B /  469MB            

runs/explore_encdec/final_model.pt: downloading bytes:           |  0.00B            

runs/explore_shared/final_model.pt: reconstructing file:   0%|          |  0.00B /  235MB            

runs/explore_shared/final_model.pt: downloading bytes:           |  0.00B            

Phase-1 checkpoints ready
[06:14:05][_smoke] backbone explore_shared (shared) | method lorma | bidir 1 | params 59.15M trainable 0.420M (0.71%)
[06:14:11][_smoke] class weights: [0.78, 1.391]
[06:14:11][_smoke] 4 docs/step | 16 steps/epoch | planned 3 steps | budget 35 min
[06:14:17][_smoke] step 3/3 | dev macro-F1 0.6190 at threshold 0.21 (at 0.50: 0.3263) acc 0.6250 | 0.1 min
[06:14:17][_smoke] training done: 3 steps in 0.1 min (dev evaluation 0.0 min) | best step 3
[06:15:12][_smoke] DEV macro-F1 0.6190 (P 0.6154 R 0.7273, acc 0.6250) at threshold 0.21 | at 0.50: 0.3263 | confusion [[TN FP][FN TP]] [[16, 15], [9, 24]]
[06:15:12][_smoke] TEST: 1517 cases, predicted 1 for 0.554 -> /tmp/cs781/p4/smoke/_smoke/submission.zip
[06:15:12][_smoke] DONE in 1.2 min
safety test passed in 77s

--- 5 min | done [] | failed []
   cj_llama_lorma_last        [06:20:05][cj_llama_lorma_last] step 550/3709 loss 0.6195 | 4.7 min
   cj_llama_lorma_hier        [06:19:58][cj_llama_lorma_hier] step 700/7064

In [9]:
# ===================== P4 CELL 8 - Results table, error analysis (FP / FN), submission zips =====================
import numpy as np
ZIPS = f"{WORK}/zips"; os.makedirs(ZIPS, exist_ok=True)
rows = []
for n, _ in RUNS if RUNS else []:
    p = f"{OUT}/{n}/summary.json"
    if not os.path.exists(p): continue
    rows.append(json.load(open(p)))
    if os.path.exists(f"{OUT}/{n}/submission.zip"):
        shutil.copy(f"{OUT}/{n}/submission.zip", f"{ZIPS}/{n}.zip")
rows.sort(key=lambda s: -s["dev_f1"])
print(f"{'run':26s} {'model':7s} {'method':8s} {'rep':11s} {'params':>7s} {'trainable':>9s} {'devF1':>7s} {'@0.5':>6s} {'thr':>4s} "
      f"{'P(1)':>6s} {'R(1)':>6s} {'acc':>6s} {'steps':>6s} {'train':>6s} {'GPU':>5s} {'test s':>6s} {'test 1%':>7s}")
for s in rows:
    m = s["method"] + ("+KD" if s.get("kd_teacher") else "")
    print(f"{s['run']:26s} {s['arch']:7s} {m:8s} {s['rep']:11s} {s['params_total'] / 1e6:6.1f}M {s['params_trainable'] / 1e6:8.3f}M "
          f"{s['dev_f1']:7.4f} {s['dev_f1_at_0.5']:6.4f} {s['threshold']:4.2f} {s['dev_p']:6.3f} {s['dev_r']:6.3f} {s['dev_acc']:6.3f} "
          f"{s['best_step']:>3d}/{s['steps']:<3d} {s['train_minutes']:5.1f}m {s['peak_gpu_gb']:5.2f} {s['infer_test_seconds']:6.0f} "
          f"{100 * s['test_pos_rate']:6.1f}%")
print("\ndevF1 = macro-F1 on dev (official primary metric) at the threshold chosen on dev; @0.5 = with the default threshold 0.5;"
      " P(1) / R(1) = precision / recall of label 1 (official secondary metrics)")
print("confusion matrices on dev  [[TN FP] [FN TP]]:")
for s in rows:
    print(f"   {s['run']:26s} {s['dev_confusion']}")
if rows:   # characteristics of false positives / negatives of the best single model, by document length
    b = rows[0]; dp = json.load(open(f"{OUT}/{b['run']}/dev_prediction.json")) if os.path.exists(f"{OUT}/{b['run']}/dev_prediction.json") else []
    z = np.load(f"{DATA}/dev.npz"); L = dict(zip([str(x) for x in z["name"]], np.diff(z["off"])))
    if dp:
        print(f"\nerror analysis of {b['run']} on dev, by length (tokens):")
        for lo, hi in [(0, 512), (512, 2048), (2048, 4096), (4096, 8192), (8192, 10 ** 9)]:
            g = [x for x in dp if lo <= L[x["id"]] < hi]
            if not g: continue
            fp = sum(x["pred"] == 1 and x["label"] == 0 for x in g); fn = sum(x["pred"] == 0 and x["label"] == 1 for x in g)
            ok = sum(x["pred"] == x["label"] for x in g)
            print(f"   {lo:>5d}-{hi if hi < 10 ** 9 else 'max':>5}: {len(g):5d} cases | acc {ok / len(g):.3f} | FP {fp:4d} | FN {fn:4d}")
        conf = [abs(x["prob1"] - b["threshold"]) for x in dp]
        wrong = [abs(x["prob1"] - b["threshold"]) for x in dp if x["pred"] != x["label"]]
        print(f"   mean distance from the threshold: all {np.mean(conf):.3f} | errors {np.mean(wrong) if wrong else 0:.3f}")
# ---- cross-check with the OFFICIAL evaluator (evaluate_phase4.py) on dev: must equal devF1 above ----
print("\nofficial evaluate_phase4.py on dev (truth = dev labels, prediction = each run's dev_prediction.json):")
for s in rows:
    dp = f"{OUT}/{s['run']}/dev_prediction.json"
    if os.path.exists(dp):
        r = subprocess.run([sys.executable, f"{WORK}/evaluate_phase4.py", "--truth", f"{DATA}/dev_truth.json", "--prediction", dp],
                           capture_output=True, text=True)
        out = " | ".join(l.strip() for l in (r.stdout + r.stderr).splitlines())
        print(f"   {s['run']:26s} {out}")
json.dump(rows, open(f"{P4}/p4_results.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{P4}/p4_results.json", path_in_repo="phase4/results/p4_results.json", repo_id=CKPT_REPO, repo_type="model")
print("\nresults table uploaded to Hugging Face: phase4/results/p4_results.json")
print("zips ready in", ZIPS, ":", sorted(os.listdir(ZIPS)))


run                        model   method   rep          params trainable   devF1   @0.5  thr   P(1)   R(1)    acc  steps  train   GPU test s test 1%
cj_llama_partial_last      llama   partial  last         107.9M   27.632M  0.7156 0.7069 0.46  0.740  0.666  0.716 4038/4040  23.5m  2.59     18   46.5%
cj_llama_full_last         llama   full     last         107.9M  107.860M  0.7137 0.6711 0.40  0.767  0.622  0.716 1928/2892  31.8m  7.11     18   40.2%
cj_shared_kd_last          shared  full+KD  last          58.8M   58.830M  0.7020 0.6778 0.44  0.759  0.602  0.705 2700/2703  26.5m  5.95     16   41.5%
cj_llama_adapter_last      llama   adapter  last         110.2M    2.479M  0.6990 0.6798 0.45  0.754  0.600  0.702 3312/3315  27.2m  4.50     16   40.7%
cj_llama_lorma_last_nocw   llama   lorma    last         108.6M    0.887M  0.6958 0.6875 0.45  0.674  0.765  0.697 1521/3045  27.2m  4.74     16   58.5%
cj_llama_lorma_last        llama   lorma    last         108.6M    0.887M  0.6849 0.6

In [10]:
# ===================== P4 CELL 9 - Ensembles (plain top-K averages + greedy), threshold chosen on dev =====================
r = subprocess.run([sys.executable, f"{WORK}/p4ens.py", "--data_dir", DATA, "--out_dir", OUT, "--hf_repo", CKPT_REPO],
                   capture_output=True, text=True)
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "[ensemble]" in l or "Error" in l or "Traceback" in l][-20:]))
if r.returncode != 0:
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
for o in sorted(os.listdir(OUT)):
    if o.startswith("ens_") and os.path.exists(f"{OUT}/{o}/submission.zip"):
        shutil.copy(f"{OUT}/{o}/submission.zip", f"{ZIPS}/{o}.zip")
print("zips:", sorted(os.listdir(ZIPS)))


[ensemble] 15 candidates: cj_llama_partial_last=0.7156, cj_llama_full_last=0.7137, cj_shared_kd_last=0.7020, cj_llama_adapter_last=0.6990, cj_llama_lorma_last_nocw=0.6958, cj_llama_lorma_last=0.6849, cj_llama_lorma_firstlast=0.6720, cj_llama_lorma_spread=0.6717, cj_llama_bitfit_last=0.6613, cj_shared_full_last=0.6463, cj_encdec_lorma_last=0.6291, cj_llama_lorma_hier=0.6268, cj_llama_head_last=0.6227, cj_gpt2_lorma_last=0.6006, cj_llama_lorma_first=0.5666
[ensemble] ens_top3: 3 models ['cj_llama_partial_last', 'cj_llama_full_last', 'cj_shared_kd_last'] | dev macro-F1 0.7312 at threshold 0.40 | confusion [[377, 120], [147, 350]] -> /kaggle/working/p4runs/ens_top3/submission.zip
[ensemble] ens_top5: 5 models ['cj_llama_partial_last', 'cj_llama_full_last', 'cj_shared_kd_last', 'cj_llama_adapter_last', 'cj_llama_lorma_last_nocw'] | dev macro-F1 0.7360 at threshold 0.42 | confusion [[386, 111], [151, 346]] -> /kaggle/working/p4runs/ens_top5/submission.zip
[ensemble] ens_greedy: 2 models ['cj